In [9]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import html
import emoji
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import pandas as pd
from langdetect import detect

#only run once
# nltk.download('punkt')
# nltk.download('stop_words')
# nltk.download('wordnet')

In [10]:
import pandas as pd
import re
import string
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

class BaseTextCleaner:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def preprocess_text(self, text):
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'#\w+', '', text)
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))

        emoji_pattern = re.compile("["                 
            u"\U0001F600-\U0001F64F"
            u"\U0001F300-\U0001F5FF"
            u"\U0001F680-\U0001F6FF"
            u"\U0001F1E0-\U0001F1FF"
            u"\U00002700-\U000027BF"
            u"\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE)
        
        text = emoji_pattern.sub(r'', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def remove_stopwords(self, text):
        return ' '.join([word for word in text.split() if word not in self.stop_words])

    def lemmatize_text(self, text):
        return ' '.join([self.lemmatizer.lemmatize(word) for word in text.split()])

    def is_english(self, text):
        return len(text.strip()) > 5 and detect(text) == 'en'

    def _is_english(self, text):
        try:
            return detect(text) == 'en'
        except:
            return False


In [11]:

class TweetCleaner(BaseTextCleaner):
    def __init__(self, filepath, encoding='latin1'):
        super().__init__()  
        self.df = pd.read_csv(filepath, encoding=encoding)

    def handle_missing_values(self):
        self.df = self.df.dropna(subset=['tweet_text'])
        self.df['emotion_in_tweet_is_directed_at'] = self.df['emotion_in_tweet_is_directed_at'].fillna('Unknown')
        return self.df

    def full_text_processing(self):
        # Handle missing values and duplicates
        self.df = self.df.dropna(subset=['tweet_text'])
        self.df = self.df.drop_duplicates(subset=['tweet_text'])

        # Filter English tweets
        self.df = self.df[self.df['tweet_text'].apply(self._is_english)]

        # Remove garbled characters
        self.df = self.df[~self.df['tweet_text'].str.contains('�')]

        # Clean text step by step
        self.df['processed_text'] = self.df['tweet_text'].astype(str).apply(self.preprocess_text)
        self.df['processed_text'] = self.df['processed_text'].apply(self.remove_stopwords)
        self.df['processed_text'] = self.df['processed_text'].apply(self.lemmatize_text)

        # Ensure target column is filled
        self.df['emotion_in_tweet_is_directed_at'] = self.df['emotion_in_tweet_is_directed_at'].fillna('Unknown')
        return self.df


In [12]:
cleaner = TweetCleaner('judge-1377884607_tweet_product_company.csv', encoding='latin1')
cleaned_df = cleaner.full_text_processing()
cleaned_df = cleaner.handle_missing_values()
print(cleaned_df.head())


                                          tweet_text  \
0  .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1  @jessedee Know about @fludapp ? Awesome iPad/i...   
2  @swonderlin Can not wait for #iPad 2 also. The...   
3  @sxsw I hope this year's festival isn't as cra...   
4  @sxtxstate great stuff on Fri #SXSW: Marissa M...   

  emotion_in_tweet_is_directed_at  \
0                          iPhone   
1              iPad or iPhone App   
2                            iPad   
3              iPad or iPhone App   
4                          Google   

  is_there_an_emotion_directed_at_a_brand_or_product  \
0                                   Negative emotion   
1                                   Positive emotion   
2                                   Positive emotion   
3                                   Negative emotion   
4                                   Positive emotion   

                                      processed_text  
0  g iphone hr tweeting dead need upgrade plugin